# FPL Season Winners Archive

Archive full season data for top N managers using the official FPL API.

## Features
- Fetch top N managers by overall rank
- Archive all 38 gameweeks of picks, captaincy, chips, transfers
- Generate strategy analysis (captaincy patterns, chip usage, squad consistency)
- Compare multiple top managers


In [ ]:
import requests
import json
import os
from collections import defaultdict, Counter
from datetime import datetime
from pathlib import Path
import pandas as pd

# Configuration
FPL_API_BASE = 'https://fantasy.premierleague.com/api'
OUTPUT_BASE = '/Users/bentindal/Desktop/coding/FPL-Auto/data'

print("Environment loaded. Ready to archive FPL season data.")

## Step 1: Fetch Top Managers

In [ ]:
def get_top_managers(num_managers=30):
    """
    Fetch top N managers by overall rank from the classic league.
    Returns list of (rank, manager_id, name, team_name, points)
    """
    print(f"Fetching top {num_managers} managers...")
    
    managers = []
    page = 1
    
    while len(managers) < num_managers:
        try:
            url = f'{FPL_API_BASE}/leagues-classic/1/standings/'
            response = requests.get(url, params={'page_standings': page})
            response.raise_for_status()
            
            data = response.json()
            results = data.get('standings', {}).get('results', [])
            
            if not results:
                break
            
            for result in results:
                if len(managers) >= num_managers:
                    break
                managers.append({
                    'rank': result['rank'],
                    'manager_id': result['entry'],
                    'player_name': result['player_name'],
                    'team_name': result['entry_name'],
                    'points': result['total']
                })
            
            page += 1
            
        except Exception as e:
            print(f"Error fetching page {page}: {e}")
            break
    
    return managers

# Fetch top 30 managers
top_managers = get_top_managers(30)

print(f"\nFound {len(top_managers)} managers:")
for mgr in top_managers[:10]:
    print(f"  Rank {mgr['rank']:3d}: {mgr['player_name']:25s} ({mgr['team_name']:20s}) - {mgr['points']} pts")
print(f"  ... and {len(top_managers) - 10} more")

## Step 2: Archive Each Manager's Season Data

In [ ]:
def archive_manager_season(manager_id, manager_name, season='2025-26', num_gws=38):
    """
    Fetch and archive all gameweek picks for a single manager.
    Returns dict with GW data and summary stats.
    """
    output_dir = Path(OUTPUT_BASE) / f'season_winners_{season}' / str(manager_id)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    gw_data = {}
    failed_gws = []
    
    for gw in range(1, num_gws + 1):
        try:
            url = f'{FPL_API_BASE}/entry/{manager_id}/event/{gw}/picks/'
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            
            data = response.json()
            gw_data[gw] = data
            
            # Save individual GW file
            gw_file = output_dir / f'gw{gw:02d}.json'
            with open(gw_file, 'w') as f:
                json.dump(data, f, indent=2)
        
        except Exception as e:
            failed_gws.append((gw, str(e)))
    
    # Save complete season
    archive_file = output_dir / '_complete_season.json'
    with open(archive_file, 'w') as f:
        json.dump(gw_data, f, indent=2)
    
    return gw_data, output_dir, failed_gws

# Archive first 3 managers (to test)
print("Archiving managers (sample: first 3 of 30)...\n")

manager_archives = {}
for mgr in top_managers[:3]:
    print(f"Archiving {mgr['player_name']} (ID: {mgr['manager_id']})...")
    gw_data, output_dir, failed = archive_manager_season(mgr['manager_id'], mgr['player_name'])
    
    if failed:
        print(f"  ⚠ Failed GWs: {failed}")
    else:
        print(f"  ✓ Archived {len(gw_data)}/38 gameweeks")
    
    manager_archives[mgr['manager_id']] = {
        'gw_data': gw_data,
        'output_dir': output_dir,
        'metadata': mgr
    }

print(f"\n✓ Archived {len(manager_archives)} managers")

## Step 3: Generate Strategy Analysis

In [ ]:
def analyze_manager_strategy(gw_data, mgr_metadata):
    """
    Analyze a manager's strategy from their full season data.
    Returns dict with captaincy, chips, squad, transfer stats.
    """
    captaincy_counts = Counter()
    chip_usage = defaultdict(list)
    player_appearances = Counter()
    transfer_data = []
    
    for gw, data in gw_data.items():
        picks = data.get('picks', [])
        
        # Captaincy
        for pick in picks:
            if pick.get('is_captain'):
                captaincy_counts[pick['element']] += 1
            player_appearances[pick['element']] += 1
        
        # Chips
        if data.get('active_chip') and data['active_chip'] != 'None':
            chip_usage[data['active_chip']].append(gw)
        
        # Transfers
        if data.get('transfers_made'):
            transfer_data.append({
                'gw': gw,
                'count': data['transfers_made'],
                'cost': data.get('transfer_cost', 0)
            })
    
    analysis = {
        'manager': mgr_metadata,
        'captaincy': {
            'top_choices': dict(captaincy_counts.most_common(10)),
            'total_gameweeks': len(gw_data)
        },
        'chip_usage': {
            chip: {
                'gameweeks': gws,
                'count': len(gws)
            }
            for chip, gws in chip_usage.items()
        },
        'squad': {
            'total_unique_players': len(player_appearances),
            'most_selected': dict(player_appearances.most_common(15))
        },
        'transfers': {
            'total': sum(t['count'] for t in transfer_data),
            'gameweeks_with_transfers': len(transfer_data),
            'avg_per_transfer_gw': round(sum(t['cost'] for t in transfer_data) / len(transfer_data), 2) if transfer_data else 0
        }
    }
    
    return analysis

# Analyze archived managers
analyses = {}
for mgr_id, archive in manager_archives.items():
    print(f"Analyzing {archive['metadata']['player_name']}...")
    analysis = analyze_manager_strategy(archive['gw_data'], archive['metadata'])
    analyses[mgr_id] = analysis
    
    # Save analysis
    analysis_file = archive['output_dir'] / 'analysis.json'
    with open(analysis_file, 'w') as f:
        json.dump(analysis, f, indent=2, default=str)
    
    print(f"  ✓ Top captain: Player {list(analysis['captaincy']['top_choices'].keys())[0]}")

print(f"\n✓ Analyzed {len(analyses)} managers")

## Step 4: Comparative Analysis

In [ ]:
# Compare top managers
print("\n" + "="*80)
print("COMPARATIVE ANALYSIS - TOP MANAGERS")
print("="*80)

comparison_data = []
for mgr_id, analysis in analyses.items():
    mgr = analysis['manager']
    comp = {
        'Rank': mgr['rank'],
        'Manager': mgr['player_name'],
        'Team': mgr['team_name'],
        'Points': mgr['points'],
        'Top Captain': list(analysis['captaincy']['top_choices'].keys())[0],
        'Unique Players': analysis['squad']['total_unique_players'],
        'Total Transfers': analysis['transfers']['total'],
        'Chips Used': len(analysis['chip_usage'])
    }
    comparison_data.append(comp)

df = pd.DataFrame(comparison_data)
print("\n" + df.to_string(index=False))

# Chip usage summary
print("\n" + "="*80)
print("CHIP USAGE PATTERNS")
print("="*80)

all_chip_usage = defaultdict(list)
for mgr_id, analysis in analyses.items():
    for chip, data in analysis['chip_usage'].items():
        all_chip_usage[chip].extend(data['gameweeks'])

for chip in ['wildcard', 'freehit', 'bboost', '3xc']:
    if chip in all_chip_usage:
        gws = all_chip_usage[chip]
        print(f"\n{chip.upper()}:")
        print(f"  Used in GWs: {sorted(set(gws))}")
        print(f"  Frequency: {len(gws)} uses across {len([m for m in manager_archives])} managers")

## Step 5: Archive ALL 30 Managers (Full Run)

In [ ]:
print("Starting full archive of top 30 managers...")
print("This will take a few minutes.\n")

all_analyses = {}
failed_managers = []

for i, mgr in enumerate(top_managers, 1):
    try:
        # Archive
        gw_data, output_dir, failed_gws = archive_manager_season(
            mgr['manager_id'], 
            mgr['player_name']
        )
        
        if len(gw_data) < 38:
            print(f"[{i:2d}/30] {mgr['player_name']:25s} - ⚠ Only {len(gw_data)}/38 GWs")
        else:
            # Analyze
            analysis = analyze_manager_strategy(gw_data, mgr)
            analysis_file = output_dir / 'analysis.json'
            with open(analysis_file, 'w') as f:
                json.dump(analysis, f, indent=2, default=str)
            
            all_analyses[mgr['manager_id']] = analysis
            print(f"[{i:2d}/30] {mgr['player_name']:25s} - ✓ ({mgr['points']} pts, rank {mgr['rank']})")
    
    except Exception as e:
        print(f"[{i:2d}/30] {mgr['player_name']:25s} - ✗ Error: {str(e)[:50]}")
        failed_managers.append((mgr['manager_id'], mgr['player_name'], str(e)))

print(f"\n✓ Completed {len(all_analyses)}/30 managers")
if failed_managers:
    print(f"⚠ Failed: {len(failed_managers)} managers")

## Step 6: Generate Master Summary

In [ ]:
def generate_master_summary(analyses, top_managers):
    """
    Create a comprehensive summary across all archived managers.
    """
    summary = {
        'metadata': {
            'timestamp': datetime.now().isoformat(),
            'season': '2025-26',
            'managers_archived': len(analyses),
            'gameweeks': 38
        },
        'top_managers': [m for m in top_managers if m['manager_id'] in analyses],
        'captaincy_meta': {
            'most_captained_overall': Counter(),
            'captaincy_variance': {}
        },
        'chip_patterns': {},
        'squad_patterns': {
            'avg_unique_players': 0,
            'most_common_players': Counter()
        }
    }
    
    # Aggregate stats
    unique_counts = []
    for mgr_id, analysis in analyses.items():
        # Captaincy
        for player_id, count in analysis['captaincy']['top_choices'].items():
            summary['captaincy_meta']['most_captained_overall'][player_id] += count
        
        # Squad
        unique_counts.append(analysis['squad']['total_unique_players'])
        for player_id in analysis['squad']['most_selected'].keys():
            summary['squad_patterns']['most_common_players'][player_id] += 1
    
    summary['captaincy_meta']['most_captained_overall'] = dict(
        summary['captaincy_meta']['most_captained_overall'].most_common(10)
    )
    summary['squad_patterns']['avg_unique_players'] = round(sum(unique_counts) / len(unique_counts), 1) if unique_counts else 0
    summary['squad_patterns']['most_common_players'] = dict(
        summary['squad_patterns']['most_common_players'].most_common(15)
    )
    
    return summary

# Generate and save master summary
if all_analyses:
    master_summary = generate_master_summary(all_analyses, top_managers)
    
    summary_file = Path(OUTPUT_BASE) / 'season_winners_2025-26' / 'MASTER_SUMMARY.json'
    summary_file.parent.mkdir(parents=True, exist_ok=True)
    
    with open(summary_file, 'w') as f:
        json.dump(master_summary, f, indent=2, default=str)
    
    print(f"✓ Master summary saved to {summary_file}")
    print(f"\nKey insights:")
    print(f"  - Managers archived: {master_summary['metadata']['managers_archived']}")
    print(f"  - Avg unique players per manager: {master_summary['squad_patterns']['avg_unique_players']}")
    print(f"  - Most captained players (top 5): {list(master_summary['captaincy_meta']['most_captained_overall'].keys())[:5]}")

## Archive Structure

```
data/
  season_winners_2025-26/
    MASTER_SUMMARY.json          # Aggregated stats across all 30 managers
    {manager_id}/
      gw01.json
      gw02.json
      ...
      gw38.json
      _complete_season.json       # All GWs in one file
      analysis.json               # Strategy analysis for this manager
```


## Reusability Notes

To use this notebook for a different season or subset of managers:

1. **Change the season**: Modify `FPL_API_BASE` and output paths
2. **Archive fewer managers**: Change `get_top_managers(30)` to any number
3. **Archive specific manager**: Call `archive_manager_season(manager_id, name)` directly
4. **Change season gameweeks**: Modify `num_gws=38` parameter
